# Pipeline ML — Ataque Cardíaco

1. **Pré-processamento** (`Heart Attack.csv`)
2. **Extração padrão** — comparação dos 6 modelos clássicos (LR, LDA, KNN, CART, NB, SVM)
3. **Redes neurais** — comparação de 8 arquiteturas MLP
4. Exportação dos artefatos usados pela API

**Modelo em produção na API:** CART (`modelo_cart_melhor.json`)

## 1. Carregando os dados

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Execute o notebook a partir da pasta api/ (ou ajuste BASE_DIR)
BASE_DIR = Path.cwd() if (Path.cwd() / 'Heart Attack.csv').exists() else Path.cwd() / 'api'
CSV_RAW = BASE_DIR / 'Heart Attack.csv'

df = pd.read_csv(CSV_RAW, sep=',')
print(f'Base carregada: {df.shape[0]} amostras, {df.shape[1]} colunas')
df.head()

## 2. Pré-processamento

In [ ]:
ATRIBUTOS = ['age', 'gender', 'impluse', 'pressurehight', 'pressurelow', 'glucose', 'kcm', 'troponin']
ALVO = 'class'

def trata_faltantes(df):
    for col in ATRIBUTOS:
        if df[col].isnull().any():
            for classe in df[ALVO].unique():
                mask = df[ALVO] == classe
                media = df.loc[mask & df[col].notnull(), col].mean()
                df.loc[mask & df[col].isnull(), col] = media
    return df

def del_inconsistencias(df):
    return df.drop_duplicates(subset=ATRIBUTOS, keep=False)

def remove_outliers(df):
    out = df.copy()
    for atributo in ATRIBUTOS:
        q75, q25 = np.percentile(out[atributo], [75, 25])
        iqr = q75 - q25
        out = out[(out[atributo] <= q75 + 1.5 * iqr) & (out[atributo] >= q25 - 1.5 * iqr)]
    return out

def normalizar(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0, ddof=1)
    sigma = np.where(sigma == 0, 1, sigma)
    return (X - mu) / sigma, mu, sigma

df = trata_faltantes(df)
df = df.drop_duplicates(keep='first')
df = del_inconsistencias(df)
print('Amostras antes de outliers:', len(df))
df = remove_outliers(df)
print('Amostras após outliers:', len(df))

In [ ]:
X_final = df[ATRIBUTOS].values.astype(float)
X_norm, mu_final, sigma_final = normalizar(X_final)

df_processado = df.copy()
df_processado[ATRIBUTOS] = X_norm

df_processado.to_csv(BASE_DIR / 'heart_attack_processado.csv', index=False)

params_json = {
    'atributos': ATRIBUTOS,
    'mu': mu_final.tolist(),
    'sigma': sigma_final.tolist(),
    'classes': ['negative', 'positive'],
    'total_amostras': len(df_processado),
}
(BASE_DIR / 'parametros_preprocessamento.json').write_text(
    json.dumps(params_json, indent=2), encoding='utf-8'
)

print('Gerado: heart_attack_processado.csv, parametros_preprocessamento.json')
df_processado.head()

## 3. Extração padrão — modelos clássicos

LR · LDA · KNN · CART · NB · SVM

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

X = df_processado[ATRIBUTOS].values.astype(float)
y = (df_processado[ALVO].values == 'positive').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models_classicos = [
    ('LR', LogisticRegression(max_iter=2000, solver='liblinear', random_state=42)),
    ('LDA', LinearDiscriminantAnalysis()),
    ('KNN', KNeighborsClassifier()),
    ('CART', DecisionTreeClassifier(random_state=42)),
    ('NB', GaussianNB()),
    ('SVM', SVC(gamma='scale', kernel='rbf', random_state=42)),
]

resultados_classicos = []
modelo_cart = None

for nome, modelo in models_classicos:
    cv = cross_val_score(modelo, X_train, y_train, cv=skf, scoring='accuracy')
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados_classicos.append({
        'modelo': nome,
        'cv_mean': float(cv.mean()),
        'cv_std': float(cv.std()),
        'acuracia_teste': float(accuracy_score(y_test, pred)),
        'f1_weighted': float(f1_score(y_test, pred, average='weighted')),
        'precisao_weighted': float(precision_score(y_test, pred, average='weighted', zero_division=0)),
        'recall_weighted': float(recall_score(y_test, pred, average='weighted')),
    })
    if nome == 'CART':
        modelo_cart = modelo

df_classicos = pd.DataFrame(resultados_classicos).sort_values(
    'acuracia_teste', ascending=False
).reset_index(drop=True)
df_classicos

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(df_classicos['modelo'], df_classicos['acuracia_teste'])
plt.ylim(0.7, 1.0)
plt.title('Comparativo — modelos clássicos (acurácia no teste)')
plt.xlabel('Modelo')
plt.ylabel('Acurácia')
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

melhor_classico = df_classicos.iloc[0]
print('Melhor no comparativo clássico:', melhor_classico['modelo'])
print('Acurácia:', round(melhor_classico['acuracia_teste'] * 100, 2), '%')
print('Modelo aplicado na API: CART')

In [ ]:
def export_cart_model(model, atributos):
    tree = model.tree_
    return {
        'atributos': atributos,
        'classes': ['negative', 'positive'],
        'tree': {
            'children_left': tree.children_left.tolist(),
            'children_right': tree.children_right.tolist(),
            'feature': tree.feature.tolist(),
            'threshold': tree.threshold.tolist(),
            'value': tree.value.reshape(tree.value.shape[0], tree.value.shape[2]).tolist(),
        },
    }

payload_classico = {
    'dataset': 'heart_attack_processado.csv',
    'amostras': len(df_processado),
    'atributos': ATRIBUTOS,
    'split': {'train': int(len(X_train)), 'test': int(len(X_test))},
    'resultados': df_classicos.to_dict(orient='records'),
    'melhor_modelo': melhor_classico.to_dict(),
    'modelo_escolhido_para_api': 'CART',
}
(BASE_DIR / 'model_results_classic.json').write_text(
    json.dumps(payload_classico, indent=2), encoding='utf-8'
)
(BASE_DIR / 'modelo_cart_melhor.json').write_text(
    json.dumps(export_cart_model(modelo_cart, ATRIBUTOS)), encoding='utf-8'
)
print('Salvo: model_results_classic.json, modelo_cart_melhor.json')

## 4. Redes neurais (MLP)

In [ ]:
from sklearn.neural_network import MLPClassifier

configuracoes = [
    ('MLP_8', (8,), 'relu'),
    ('MLP_16', (16,), 'relu'),
    ('MLP_32', (32,), 'relu'),
    ('MLP_16_8', (16, 8), 'relu'),
    ('MLP_32_16', (32, 16), 'relu'),
    ('MLP_64_32', (64, 32), 'relu'),
    ('MLP_32_TANH', (32,), 'tanh'),
    ('MLP_64_32_TANH', (64, 32), 'tanh'),
]

resultados_nn = []
melhor_modelo_nn = None
melhor_acc_nn = -1

for nome, hidden, activation in configuracoes:
    modelo = MLPClassifier(
        hidden_layer_sizes=hidden,
        activation=activation,
        solver='adam',
        max_iter=2000,
        early_stopping=True,
        random_state=42,
    )
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, pred)
    resultados_nn.append({
        'modelo': nome,
        'hidden_layers': list(hidden),
        'activation': activation,
        'acuracia': float(acc),
        'f1_weighted': float(f1_score(y_test, pred, average='weighted')),
    })
    if acc > melhor_acc_nn:
        melhor_acc_nn = acc
        melhor_modelo_nn = modelo

df_nn = pd.DataFrame(resultados_nn).sort_values('acuracia', ascending=False).reset_index(drop=True)
df_nn

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(df_nn['modelo'], df_nn['acuracia'])
plt.ylim(0.65, 1.0)
plt.title('Comparativo — redes neurais (acurácia no teste)')
plt.xlabel('Modelo')
plt.ylabel('Acurácia')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

melhor_nn = df_nn.iloc[0]
print('Melhor rede neural:', melhor_nn['modelo'])
print('Acurácia:', round(melhor_nn['acuracia'] * 100, 2), '%')

In [ ]:
payload_nn = {
    'dataset': 'heart_attack_processado.csv',
    'resultados': df_nn.to_dict(orient='records'),
    'melhor_modelo': melhor_nn.to_dict(),
}
(BASE_DIR / 'model_results_neural_networks.json').write_text(
    json.dumps(payload_nn, indent=2), encoding='utf-8'
)

modelo_nn_export = {
    'atributos': ATRIBUTOS,
    'classes': ['negative', 'positive'],
    'hidden_layers': list(melhor_modelo_nn.hidden_layer_sizes),
    'activation': melhor_modelo_nn.activation,
    'coefs': [c.tolist() for c in melhor_modelo_nn.coefs_],
    'intercepts': [b.tolist() for b in melhor_modelo_nn.intercepts_],
}
(BASE_DIR / 'modelo_neural_melhor.json').write_text(
    json.dumps(modelo_nn_export), encoding='utf-8'
)
print('Salvo: model_results_neural_networks.json, modelo_neural_melhor.json')

## 5. Resumo

| Etapa | Arquivo gerado |
|-------|----------------|
| Pré-processamento | `heart_attack_processado.csv`, `parametros_preprocessamento.json` |
| Modelos clássicos | `model_results_classic.json`, **`modelo_cart_melhor.json`** (API) |
| Redes neurais | `model_results_neural_networks.json`, `modelo_neural_melhor.json` |

A API Flutter usa **CART** em `POST /diagnosticos/risco`.